# NER Benchmark: Ollama LLM vs Hugging Face Model

Автономный бенчмарк для сравнения точности NER между двумя подходами:
- `Ollama` LLM (через `/api/chat`)
- специализированная NER-модель из Hugging Face (`transformers.pipeline`)

Golden set хранится отдельно: `ner_benchmark/golden_set/ner_golden.jsonl`.

In [ ]:
# %pip install transformers torch httpx

In [ ]:
from __future__ import annotations

import csv
import json
import statistics
import time
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import httpx
from transformers import pipeline

In [ ]:
GOLDEN_PATH = Path('ner_benchmark/golden_set/ner_golden.jsonl')
RUNS_DIR = Path('ner_benchmark/runs')
ALLOWED_LABELS = {'PER', 'ORG', 'LOC'}

HF_MODEL_NAME = 'Babelscape/wikineural-multilingual-ner'
OLLAMA_BASE_URL = 'http://localhost:11434'
OLLAMA_MODEL_NAME = 'qwen3:14b'

RUN_HF = True
RUN_OLLAMA = True

MAX_SAMPLES = None  # Например: 10
REQUEST_TIMEOUT_SEC = 120.0
MAX_ATTEMPTS = 3

In [ ]:
@dataclass(slots=True)
class Entity:
    text: str
    label: str


@dataclass(slots=True)
class Sample:
    sample_id: str
    text: str
    entities: list[Entity]


def normalize_text(value: str) -> str:
    keep = []
    for ch in value.strip().lower():
        if ch.isalnum() or ch.isspace():
            keep.append(ch)
    return ' '.join(''.join(keep).split())


def normalize_label(value: str) -> str:
    label = value.strip().upper().replace('B-', '').replace('I-', '')
    aliases = {
        'PERSON': 'PER',
        'PER': 'PER',
        'ORGANIZATION': 'ORG',
        'ORG': 'ORG',
        'LOCATION': 'LOC',
        'GPE': 'LOC',
        'LOC': 'LOC',
    }
    return aliases.get(label, label)


def load_golden(path: Path, max_samples: int | None = None) -> list[Sample]:
    rows: list[Sample] = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                continue
            data = json.loads(line)
            entities = [
                Entity(text=str(e['text']), label=normalize_label(str(e['label'])))
                for e in data.get('entities', [])
                if normalize_label(str(e.get('label', ''))) in ALLOWED_LABELS
            ]
            rows.append(Sample(sample_id=str(data['id']), text=str(data['text']), entities=entities))
            if max_samples is not None and len(rows) >= max_samples:
                break
    if not rows:
        raise ValueError(f'Golden set is empty: {path}')
    return rows


golden_samples = load_golden(GOLDEN_PATH, MAX_SAMPLES)
print(f'Loaded samples: {len(golden_samples)}')
print('Example:', golden_samples[0])

In [ ]:
class HuggingFaceNERExtractor:
    def __init__(self, model_name: str) -> None:
        self._pipe = pipeline('token-classification', model=model_name, aggregation_strategy='simple')

    def extract(self, text: str) -> list[Entity]:
        raw = self._pipe(text)
        entities: list[Entity] = []
        for item in raw:
            label = normalize_label(str(item.get('entity_group', item.get('entity', ''))))
            if label not in ALLOWED_LABELS:
                continue
            word = str(item.get('word', '')).replace('##', '').strip()
            if not word:
                continue
            entities.append(Entity(text=word, label=label))
        return entities


class OllamaNERExtractor:
    def __init__(self, base_url: str, model_name: str, timeout_sec: float = 120.0, max_attempts: int = 3) -> None:
        self.base_url = base_url
        self.model_name = model_name
        self.timeout_sec = timeout_sec
        self.max_attempts = max_attempts

    def _build_prompt(self, text: str) -> str:
        return (
            'Extract named entities from the text.\n'
            'Allowed labels only: PER, ORG, LOC.\n'
            'Return one JSON object only with schema: '
            '{"entities": [{"text": "...", "label": "PER|ORG|LOC"}]}.\n'
            'No markdown. No extra keys.\n\n'
            f'Text:\n{text}'
        )

    def extract(self, text: str) -> list[Entity]:
        prompt = self._build_prompt(text)
        last_error: Exception | None = None

        for attempt in range(1, self.max_attempts + 1):
            try:
                with httpx.Client(base_url=self.base_url, timeout=self.timeout_sec) as client:
                    resp = client.post(
                        '/api/chat',
                        json={
                            'model': self.model_name,
                            'messages': [
                                {'role': 'system', 'content': 'You are a precise NER extraction engine.'},
                                {'role': 'user', 'content': prompt},
                            ],
                            'format': 'json',
                            'stream': False,
                            'options': {'temperature': 0.0, 'top_p': 1.0},
                        },
                    )
                    resp.raise_for_status()
                    payload = resp.json()
                    content = ((payload.get('message') or {}).get('content') or '').strip()
                    data = json.loads(content)

                    entities: list[Entity] = []
                    for item in data.get('entities', []):
                        if not isinstance(item, dict):
                            continue
                        label = normalize_label(str(item.get('label', '')))
                        text_value = str(item.get('text', '')).strip()
                        if label in ALLOWED_LABELS and text_value:
                            entities.append(Entity(text=text_value, label=label))
                    return entities
            except Exception as exc:
                last_error = exc
                if attempt < self.max_attempts:
                    time.sleep(min(2**attempt, 5))

        raise RuntimeError('Ollama extraction failed') from last_error

In [ ]:
def to_set(entities: list[Entity]) -> set[tuple[str, str]]:
    return {(normalize_label(e.label), normalize_text(e.text)) for e in entities if normalize_text(e.text)}


def score_sample(gold_entities: list[Entity], pred_entities: list[Entity]) -> dict[str, Any]:
    gold_set = to_set(gold_entities)
    pred_set = to_set(pred_entities)

    tp = len(gold_set & pred_set)
    fp = len(pred_set - gold_set)
    fn = len(gold_set - pred_set)

    exact_match = 1.0 if gold_set == pred_set else 0.0
    return {
        'tp': tp,
        'fp': fp,
        'fn': fn,
        'exact_match': exact_match,
        'gold_set': gold_set,
        'pred_set': pred_set,
    }


def prf(tp: int, fp: int, fn: int) -> tuple[float, float, float]:
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    return precision, recall, f1


def run_benchmark(name: str, extractor: Any, samples: list[Sample]) -> dict[str, Any]:
    totals = {'tp': 0, 'fp': 0, 'fn': 0}
    by_label: dict[str, dict[str, int]] = {label: {'tp': 0, 'fp': 0, 'fn': 0} for label in ALLOWED_LABELS}

    latencies_ms: list[float] = []
    exact_matches: list[float] = []
    errors = 0
    per_sample_rows: list[dict[str, Any]] = []

    for sample in samples:
        start = time.perf_counter()
        try:
            pred_entities = extractor.extract(sample.text)
        except Exception:
            pred_entities = []
            errors += 1
        elapsed_ms = (time.perf_counter() - start) * 1000.0

        latencies_ms.append(elapsed_ms)
        scored = score_sample(sample.entities, pred_entities)

        totals['tp'] += scored['tp']
        totals['fp'] += scored['fp']
        totals['fn'] += scored['fn']
        exact_matches.append(scored['exact_match'])

        for label in ALLOWED_LABELS:
            gold_label = {x for x in scored['gold_set'] if x[0] == label}
            pred_label = {x for x in scored['pred_set'] if x[0] == label}
            by_label[label]['tp'] += len(gold_label & pred_label)
            by_label[label]['fp'] += len(pred_label - gold_label)
            by_label[label]['fn'] += len(gold_label - pred_label)

        per_sample_rows.append({
            'model': name,
            'sample_id': sample.sample_id,
            'tp': scored['tp'],
            'fp': scored['fp'],
            'fn': scored['fn'],
            'exact_match': scored['exact_match'],
            'latency_ms': elapsed_ms,
        })

    micro_p, micro_r, micro_f1 = prf(totals['tp'], totals['fp'], totals['fn'])

    per_label_metrics: dict[str, dict[str, float]] = {}
    macro_precisions: list[float] = []
    macro_recalls: list[float] = []
    macro_f1s: list[float] = []

    for label in sorted(ALLOWED_LABELS):
        p, r, f1 = prf(by_label[label]['tp'], by_label[label]['fp'], by_label[label]['fn'])
        per_label_metrics[label] = {
            'precision': p,
            'recall': r,
            'f1': f1,
            'tp': by_label[label]['tp'],
            'fp': by_label[label]['fp'],
            'fn': by_label[label]['fn'],
        }
        macro_precisions.append(p)
        macro_recalls.append(r)
        macro_f1s.append(f1)

    p95_latency = statistics.quantiles(latencies_ms, n=100)[94] if len(latencies_ms) >= 2 else latencies_ms[0]

    summary = {
        'model': name,
        'num_samples': len(samples),
        'errors': errors,
        'entity_level_micro': {
            'precision': micro_p,
            'recall': micro_r,
            'f1': micro_f1,
            'tp': totals['tp'],
            'fp': totals['fp'],
            'fn': totals['fn'],
        },
        'entity_level_macro': {
            'precision': sum(macro_precisions) / len(macro_precisions),
            'recall': sum(macro_recalls) / len(macro_recalls),
            'f1': sum(macro_f1s) / len(macro_f1s),
        },
        'per_label': per_label_metrics,
        'exact_match_rate': sum(exact_matches) / len(exact_matches),
        'hallucination_rate': totals['fp'] / (totals['tp'] + totals['fp']) if (totals['tp'] + totals['fp']) > 0 else 0.0,
        'miss_rate': totals['fn'] / (totals['tp'] + totals['fn']) if (totals['tp'] + totals['fn']) > 0 else 0.0,
        'latency_ms': {
            'avg': sum(latencies_ms) / len(latencies_ms),
            'p95': p95_latency,
            'max': max(latencies_ms),
        },
    }

    return {'summary': summary, 'per_sample': per_sample_rows}

In [ ]:
results: dict[str, dict[str, Any]] = {}

if RUN_HF:
    hf_extractor = HuggingFaceNERExtractor(HF_MODEL_NAME)
    results['huggingface'] = run_benchmark('huggingface', hf_extractor, golden_samples)

if RUN_OLLAMA:
    ollama_extractor = OllamaNERExtractor(
        base_url=OLLAMA_BASE_URL,
        model_name=OLLAMA_MODEL_NAME,
        timeout_sec=REQUEST_TIMEOUT_SEC,
        max_attempts=MAX_ATTEMPTS,
    )
    results['ollama'] = run_benchmark('ollama', ollama_extractor, golden_samples)

if not results:
    raise ValueError('No benchmark runs were enabled. Set RUN_HF and/or RUN_OLLAMA to True.')

for name, data in results.items():
    print(f'=== {name.upper()} ===')
    print(json.dumps(data['summary'], ensure_ascii=False, indent=2))

In [ ]:
def save_artifacts(all_results: dict[str, dict[str, Any]], runs_dir: Path) -> Path:
    run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    run_dir = runs_dir / run_id
    run_dir.mkdir(parents=True, exist_ok=True)

    summary_payload = {
        'run_at_utc': datetime.now(timezone.utc).isoformat(),
        'golden_path': str(GOLDEN_PATH.resolve()),
        'hf_model_name': HF_MODEL_NAME,
        'ollama_model_name': OLLAMA_MODEL_NAME,
        'results': {name: data['summary'] for name, data in all_results.items()},
    }

    summary_json = run_dir / 'summary.json'
    summary_json.write_text(json.dumps(summary_payload, ensure_ascii=False, indent=2), encoding='utf-8')

    summary_csv = run_dir / 'summary.csv'
    fieldnames = [
        'model',
        'num_samples',
        'errors',
        'micro_precision',
        'micro_recall',
        'micro_f1',
        'macro_precision',
        'macro_recall',
        'macro_f1',
        'exact_match_rate',
        'hallucination_rate',
        'miss_rate',
        'latency_avg_ms',
        'latency_p95_ms',
    ]

    with summary_csv.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for name, data in all_results.items():
            s = data['summary']
            writer.writerow({
                'model': name,
                'num_samples': s['num_samples'],
                'errors': s['errors'],
                'micro_precision': s['entity_level_micro']['precision'],
                'micro_recall': s['entity_level_micro']['recall'],
                'micro_f1': s['entity_level_micro']['f1'],
                'macro_precision': s['entity_level_macro']['precision'],
                'macro_recall': s['entity_level_macro']['recall'],
                'macro_f1': s['entity_level_macro']['f1'],
                'exact_match_rate': s['exact_match_rate'],
                'hallucination_rate': s['hallucination_rate'],
                'miss_rate': s['miss_rate'],
                'latency_avg_ms': s['latency_ms']['avg'],
                'latency_p95_ms': s['latency_ms']['p95'],
            })

    per_sample_csv = run_dir / 'per_sample_metrics.csv'
    with per_sample_csv.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(
            f,
            fieldnames=['model', 'sample_id', 'tp', 'fp', 'fn', 'exact_match', 'latency_ms'],
        )
        writer.writeheader()
        for name, data in all_results.items():
            for row in data['per_sample']:
                writer.writerow(row)

    print(f'Artifacts saved to: {run_dir.resolve()}')
    return run_dir

save_artifacts(results, RUNS_DIR)